In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
model = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature=0
)

In [3]:
load_dotenv()

True

In [ ]:
# define state
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
# task - 
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [6]:
# task -2 
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [7]:
# build graph 
graph = StateGraph(JokeState)

# nodes
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

# edges
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

# check pointer
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'FIFa'}, config=config1)

{'topic': 'FIFa',
 'joke': 'Why did the FIFA player bring a ladder to the game?\n\nBecause he wanted to take his game to the next level! (get it?)',
 'explanation': 'A classic play on words. This joke is funny because it uses a common English idiom, "take it to the next level," in a literal and figurative sense.\n\nIn a figurative sense, "take it to the next level" means to improve or elevate something, such as a person\'s skills or performance, to a higher standard or degree of excellence. In the context of the joke, the FIFA player wants to improve their gaming skills and performance.\n\nHowever, the phrase "next level" also has a literal meaning, referring to a physical level or elevation, such as a higher floor or a raised platform. The joke relies on this double meaning, as the FIFA player brings a ladder to the game, which is a physical object used to climb to a higher level.\n\nThe punchline, "he wanted to take his game to the next level," is humorous because it sets up the expe

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Marriage and single people'}, config=config2)

{'topic': 'Marriage and single people',
 'joke': 'Why did the single person bring a ladder to the wedding?\n\nBecause they heard marriage was a step up from being single! (get it?)',
 'explanation': 'A clever play on words. The joke relies on a double meaning of the phrase "a step up." In a literal sense, a ladder is a tool used to climb up steps or reach higher places. However, the phrase "a step up" is also an idiomatic expression meaning an improvement or advancement in life.\n\nIn this joke, the single person brings a ladder to the wedding because they\'ve heard that marriage is "a step up" from being single, implying that marriage is a better or more desirable state. The humor comes from the fact that the single person takes this phrase literally, thinking that marriage requires a physical step up, and therefore brings a ladder to the wedding. The punchline is a lighthearted and humorous way to poke fun at the idea of marriage being a superior state, and the single person\'s misin